[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/26_Semantic_Layer_Basics.ipynb)

# 语义层基础:把知识图谱、本体和映射整合在一起

## 概述

本课将你已经接触过的三样东西 —— 知识图谱、本体和 RDF 导出 —— 整合成一个最小的*语义层*:一个其类型、关系和属性都被**显式映射**到本体术语的知识图谱,这样生成的 RDF 就可以针对共享词表使用 SPARQL 进行查询。

**文档**:[API 参考](https://semantica.readthedocs.io/concepts/)

### 🎯 学习目标

- 使用 `GraphBuilder` 构建一个小型知识图谱
- 使用 `OntologyGenerator` 根据该图生成一个起步本体
- 编写从实体类型、关系类型和属性到本体术语的**显式**映射
- 生成与本体对齐的 RDF,并使用 `TripletStore` 存储
- 用一个小小的 SPARQL 查询回答一个业务问题

### 📚 前置知识

- [07_Building_Knowledge_Graphs.ipynb](./07_Building_Knowledge_Graphs.ipynb) —— 基于实体和关系构建图
- [14_Ontology.ipynb](./14_Ontology.ipynb) —— 本体生成
- [20_Triplet_Store.ipynb](./20_Triplet_Store.ipynb) —— 三元组存储后端

> [!NOTE]
> **教学用映射 vs. 受治理的映射。**本课中的映射只是演示:它们存放在一个 Python 字典中,并由生成的本体推导而来。生产级的语义层会使用受治理的标识符、手工设计的本体、显式的数据源映射、校验(SHACL)、溯源和版本管理 —— 该工作流在[高级:手工本体 + Snowflake 映射](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb)中介绍。

## 安装

三元组存储这一步使用内嵌的 Oxigraph 后端,因此请安装对应的扩展。请至少固定 0.6.7 版本:更早的版本可能生成没有 URI 的本体类(#1103),这会让下面的映射悄无声息地失效,而不是明确地报错。

```bash
pip install "semantica[tripletstore-oxigraph]>=0.6.7"
```

---

## 步骤 1:构建知识图谱

从一小组显式的实体和关系开始 —— 两个人、一个组织和一个项目。

In [ ]:
!pip install "semantica[tripletstore-oxigraph]>=0.6.7"


In [ ]:
from semantica.kg import GraphBuilder

entities = [
    {"id": "e1", "type": "Person", "name": "Alice", "properties": {"age": 30, "role": "Engineer"}},
    {"id": "e2", "type": "Person", "name": "Bob", "properties": {"age": 35, "role": "Manager"}},
    {"id": "e3", "type": "Organization", "name": "Tech Corp", "properties": {"founded": 2010}},
    {"id": "e4", "type": "Project", "name": "Project Alpha", "properties": {"status": "active"}},
]

relationships = [
    {"source": "e1", "target": "e2", "type": "reports_to", "properties": {}},
    {"source": "e1", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e2", "target": "e3", "type": "works_for", "properties": {}},
    {"source": "e1", "target": "e4", "type": "works_on", "properties": {}},
]

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Entities ({len(knowledge_graph['entities'])}):")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']}) {entity['properties']}")

print(f"\nRelationships ({len(knowledge_graph['relationships'])}):")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

## 步骤 2:生成起步本体

`OntologyGenerator` 从图记录中推断出 OWL 类和属性。由于 `GraphBuilder` 把业务属性保存在每个实体的 `properties` 字典中,而本体推断读取的是记录字段,因此我们首先创建一个扁平的**推断视图**。知识图谱本身保持不变。这里有两个设置很重要:

- `base_uri` 把每个生成的术语放入*你自己的*命名空间
- `min_occurrences=1` 会包含只出现一次的类(默认值 2 会在这个很小的演示图中丢掉 `Organization` 和 `Project`)

注意,生成器会对名称进行规范化:关系类型 `works_for` 变成本体属性 `worksFor`。这正是下一步要**显式**映射术语,而不是靠名称匹配的原因。

In [ ]:
from semantica.ontology import OntologyGenerator

BASE_URI = "https://example.org/company/"

# Adapt the property-graph representation to the record shape consumed by
# OntologyGenerator, so age/role/founded/status become declared properties.
ontology_input = {
    "entities": [
        {
            **{key: value for key, value in entity.items() if key != "properties"},
            **entity.get("properties", {}),
        }
        for entity in knowledge_graph["entities"]
    ],
    "relationships": knowledge_graph["relationships"],
}

generator = OntologyGenerator(base_uri=BASE_URI, min_occurrences=1)
ontology = generator.generate_from_graph(ontology_input)

# OntologyGenerator calls datatype properties `data`; TripletStore's public
# ontology contract calls them `datatype`. Normalize that boundary explicitly.
store_ontology = {
    **ontology,
    "properties": [
        {**prop, "type": "datatype" if prop["type"] == "data" else prop["type"]}
        for prop in ontology["properties"]
    ],
}

print("Classes:")
for ontology_class in ontology["classes"]:
    print(f"  {ontology_class['name']:<14} {ontology_class['uri']}")

print("\nProperties:")
for prop in ontology["properties"]:
    print(f"  {prop['name']:<14} {prop['type']:<7} {prop['uri']}  "
          f"(domain={prop['domain']}, range={prop['range']})")

assert len(ontology["classes"]) == 3

## 步骤 3:将图映射到本体术语

语义层的核心是映射契约:哪个源类型、关系和属性对应哪个本体术语。

- **实体类型**和**关系类型**:每个生成的类/属性都记录了它所推断自的源名称(`metadata["inferred_from"]`),因此映射可以直接从本体本身读出 —— 无需在 `works_for` 和 `worksFor` 之间做脆弱的名称匹配。
- **属性**:扁平的推断视图使 `name`、`age`、`role`、`founded` 和 `status` 成为真正生成的数据类型属性。因此每个映射都指向本体中声明过的术语 —— 不会仅在映射时才临时捏造一个 URI。

In [ ]:
entity_type_mappings = {
    ontology_class["metadata"]["inferred_from"]: ontology_class["uri"]
    for ontology_class in ontology["classes"]
}

relationship_type_mappings = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] == "object"
}

datatype_property_uris = {
    prop["metadata"]["inferred_from"]: prop["uri"]
    for prop in ontology["properties"]
    if prop["type"] != "object"
}

property_mappings = datatype_property_uris

semantic_layer = {
    "graph": knowledge_graph,
    "ontology": ontology,
    "mappings": {
        "entity_type_mappings": entity_type_mappings,
        "relationship_type_mappings": relationship_type_mappings,
        "property_mappings": property_mappings,
    },
}

for mapping_name, mapping in semantic_layer["mappings"].items():
    print(f"{mapping_name}:")
    for source, target in mapping.items():
        print(f"  {source:<12} -> {target}")

# Every type and relationship in the graph must have an ontology term
assert set(entity_type_mappings) == {entity["type"] for entity in entities}
assert set(relationship_type_mappings) == {rel["type"] for rel in relationships}
assert set(property_mappings) == {"name", "age", "role", "founded", "status"}
assert set(property_mappings.values()) <= {prop["uri"] for prop in ontology["properties"]}

## 步骤 4:应用映射

应用语义层,意味着重写这张图,使每个类型、关系和属性键都是本体术语。被导出和存储的是这张*对齐后*的图,而不是原始的图。

In [ ]:
aligned_graph = {
    "entities": [
        {
            **entity,
            "type": entity_type_mappings[entity["type"]],
            "properties": {
                property_mappings["name"]: entity["name"],
                **{
                    property_mappings[key]: value
                    for key, value in entity["properties"].items()
                },
            },
        }
        for entity in knowledge_graph["entities"]
    ],
    "relationships": [
        {**rel, "type": relationship_type_mappings[rel["type"]]}
        for rel in knowledge_graph["relationships"]
    ],
}

print("Aligned entity sample:")
sample = aligned_graph["entities"][0]
print(f"  id:   {sample['id']}")
print(f"  type: {sample['type']}")
for key, value in sample["properties"].items():
    print(f"  {key} = {value}")

print("\nAligned relationship sample:")
print(f"  {aligned_graph['relationships'][0]['type']}")

## 步骤 5:存储并导出完整的、与本体对齐的 RDF

`TripletStore.store()` 会物化本体声明和对齐后的实例图。然后我们通过存储的公共 API 读取这些三元组,并把这张完整的 RDF 图序列化为 Turtle。这样可以避开 `RDFExporter` 紧凑的实体投影,后者不会包含实体 `properties` 字典中的任意条目。

In [ ]:
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import OWL, RDF
from semantica.triplet_store import TripletStore

store = TripletStore(backend="oxigraph")
result = store.store(aligned_graph, store_ontology)
print(f"Stored triples: {result['processed']} (failed: {result['failed']})")

rdf_graph = Graph()
for triplet in store.get_triplets():
    datatype = triplet.metadata.get("datatype")
    if datatype:
        object_term = Literal(triplet.object, datatype=URIRef(datatype))
    elif triplet.object.startswith(("http://", "https://", "urn:")):
        object_term = URIRef(triplet.object)
    else:
        object_term = Literal(triplet.object)
    rdf_graph.add((URIRef(triplet.subject), URIRef(triplet.predicate), object_term))

rdf_graph.serialize(destination="semantic_layer.ttl", format="turtle")
turtle = open("semantic_layer.ttl", encoding="utf-8").read()
print(turtle[:600])

# The exported RDF contains declarations plus mapped instance facts.
declared_datatype_properties = {
    str(subject) for subject in rdf_graph.subjects(RDF.type, OWL.DatatypeProperty)
}
assert result["failed"] == 0
assert set(property_mappings.values()) <= declared_datatype_properties
assert (
    URIRef(BASE_URI + "e1"),
    URIRef(property_mappings["role"]),
    Literal("Engineer"),
) in rdf_graph
assert (
    URIRef(BASE_URI + "e1"),
    URIRef(relationship_type_mappings["works_for"]),
    URIRef(BASE_URI + "e3"),
) in rdf_graph
print("... exported semantic_layer.ttl")

## 步骤 6:查询语义层

内嵌的 Oxigraph 后端在内存中运行,因此除了安装 `tripletstore-oxigraph` 扩展外,无需启动任何东西。组织通过其被映射的 `name` 谓语来约束;因此该查询指的就是 *Tech Corp*,而不会意外地匹配到所有组织的员工。

In [ ]:
query = f"""
SELECT ?name ?role WHERE {{
    ?person <{BASE_URI}worksFor> ?org .
    ?org <{BASE_URI}name> "Tech Corp" .
    ?person <{BASE_URI}name> ?name .
    ?person <{BASE_URI}role> ?role .
}}
ORDER BY ?name
"""
query_result = store.execute_query(query)

print("\nWho works for Tech Corp, and in which role?")
for binding in query_result.bindings:
    print(f"  {binding['name']['value']} — {binding['role']['value']}")

assert [(row["name"]["value"], row["role"]["value"]) for row in query_result.bindings] == [
    ("Alice", "Engineer"),
    ("Bob", "Manager"),
]

## 🧹 可选:清理

In [ ]:
from pathlib import Path

ttl_file = Path("semantic_layer.ttl")
if ttl_file.exists():
    ttl_file.unlink()
    print(f"Removed {ttl_file}")

## 总结

最小的语义层是一种组合,而你现在已经构建了其中的每个部分:

1. **知识图谱** —— 基于显式的实体和关系,使用 `GraphBuilder`
2. **本体** —— 使用带你的 `base_uri` 的 `OntologyGenerator`
3. **显式映射** —— 实体类型、关系类型和属性,每一个都绑定到一个本体术语
4. **与本体对齐的 RDF** —— 将映射应用到图上,通过 `TripletStore` 物化,并根据存储自身的三元组序列化为 Turtle
5. **可查询的存储** —— `TripletStore`(内嵌 Oxigraph)对共享词表回答 SPARQL 问题

### 接下来去哪

该工作流的生产版本 —— 手工设计的受治理本体、来自数据仓库的显式源到本体映射、n 元建模、SHACL 校验、溯源和版本管理 —— 在[高级:手工本体 + Snowflake 映射](../advanced/13_Manual_Ontology_Snowflake_Mapping.ipynb)中介绍。